<a href="https://colab.research.google.com/github/belinatom/NALAPROJECT/blob/main/nlp_task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2 — BERT Fine-tuning

## Models Compared
1. DistilBERT-multilingual-cased (generic multilingual baseline)
2. SwahBERT-Cased (Swahili-only)
3. AfriBERTa-Large (11 African languages including Swahili)


In [ ]:
# Cell 1 — Install & imports
!pip install -q gdown wandb mlflow sentencepiece

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW

from transformers import (
    AutoTokenizer, BertTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, classification_report

import gdown

seed = 42
np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_style('whitegrid')
print(f'✓ Imports loaded | Device: {device}')


In [ ]:
# Cell 2 — Download dataset from Google Drive
FILE_ID = '1Rp9Fg1BitlAR3d1JyH4DmTp3aSKr8H1t'
gdown.download(
    f'https://drive.google.com/uc?id={FILE_ID}',
    '/content/swahiliproverbs.csv',
    quiet=False, fuzzy=True
)

df = pd.read_csv('/content/swahiliproverbs.csv')
df = df[['swahili_proverb', 'label']].dropna().copy()
df['swahili_proverb'] = df['swahili_proverb'].astype(str).str.strip()
df['label']           = df['label'].astype(str).str.strip()
df = df[df['swahili_proverb'] != ''].reset_index(drop=True)

text_col, label_col = 'swahili_proverb', 'label'
print(f'✓ Loaded: {len(df)} proverbs, {df[label_col].nunique()} categories')
df.head(3)


In [ ]:
# Encode labels and split data
le = LabelEncoder()
y = le.fit_transform(df[label_col])
x = df[text_col].values
num_classes = len(le.classes_)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=seed, stratify=y
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.1, random_state=seed, stratify=y_train
)
print(f'✓ Train: {len(x_train)} | Val: {len(x_val)} | Test: {len(x_test)} | Classes: {num_classes}')


In [ ]:
# Config and DataLoader helper
MAX_LENGTH = 64
BATCH_SIZE = 32

def make_loader(texts, labels, tokenizer, shuffle=False):
    enc = tokenizer(
        list(texts), max_length=MAX_LENGTH,
        truncation=True, padding=True, return_tensors='pt'
    )
    dataset = TensorDataset(
        enc['input_ids'], enc['attention_mask'],
        torch.tensor(labels, dtype=torch.long)
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

print('✓ DataLoader helper defined')


In [ ]:
# Fine-tuning function
def train_classifier(model_name, label, num_epochs=5, lr=2e-5):
    print(f'\n{"="*60}\nFINE-TUNING: {label}\n{"="*60}')
    # SwahBERT ships only vocab.txt (BERT WordPiece); load it explicitly.
    # Other models resolve fine through AutoTokenizer.
    if 'swahbert' in model_name.lower():
        tokenizer = BertTokenizer.from_pretrained(model_name)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

    tr_loader = make_loader(x_train, y_train, tokenizer, shuffle=True)
    vl_loader = make_loader(x_val,   y_val,   tokenizer)
    te_loader = make_loader(x_test,  y_test,  tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_classes, ignore_mismatched_sizes=True
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = len(tr_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, total_steps)

    best_val_acc, best_val_f1 = 0, 0
    history = {'epoch': [], 'train_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for input_ids, attention_mask, labels in tr_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
        train_loss = total_loss / len(tr_loader)

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for input_ids, attention_mask, labels in vl_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                targets.extend(labels.numpy())

        val_acc = accuracy_score(targets, preds)
        val_f1  = f1_score(targets, preds, average='macro', zero_division=0)

        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc); history['val_f1'].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_acc, best_val_f1 = val_acc, val_f1
            torch.save(model.state_dict(), f'/tmp/best_{label}.pth')

        print(f'  Epoch {epoch+1}/{num_epochs} | Loss={train_loss:.4f} | Val Acc={val_acc:.4f} F1={val_f1:.4f}')

    model.load_state_dict(torch.load(f'/tmp/best_{label}.pth'))
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in te_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            targets.extend(labels.numpy())

    test_acc = accuracy_score(targets, preds)
    test_f1  = f1_score(targets, preds, average='macro', zero_division=0)
    test_mcc = matthews_corrcoef(targets, preds)

    print(f'\n✓ {label}: Val Macro-F1={best_val_f1:.4f}')
    print(f'✓ {label}: Test Acc={test_acc:.4f} | Macro-F1={test_f1:.4f} | MCC={test_mcc:.4f}')
    print(f'\nPer-class report ({label}):\n')
    print(classification_report(targets, preds, zero_division=0))

    return {'tag': label, 'val_acc': best_val_acc, 'val_f1': best_val_f1,
            'test_acc': test_acc, 'test_f1': test_f1, 'test_mcc': test_mcc,
            'history': history, 'model_path': f'/tmp/best_{label}.pth',
            'model_name': model_name, 'epochs': num_epochs, 'lr': lr}

print('✓ Training function defined')


In [ ]:
# Fine-tune DistilBERT-multilingual (10 epochs)
result_distilbert = train_classifier(
    'distilbert-base-multilingual-cased',
    label='DistilBERT-10ep', num_epochs=10, lr=3e-5
)


In [ ]:
# Fine-tune SwahBERT
import shutil

# Download SwahBERT model files
if not os.path.exists('/content/swahbert/config.json'):
    os.makedirs('/content/swahbert', exist_ok=True)
    gdown.download_folder(
        'https://drive.google.com/drive/folders/1cCcPopqTyzY6AnH9quKcT9kG5zH7tgEZ',
        output='/content/swahbert', quiet=False
    )
    # Rename config to HF standard
    if os.path.exists('/content/swahbert/swahbert_config.json'):
        shutil.copy('/content/swahbert/swahbert_config.json', '/content/swahbert/config.json')

result_swahbert = train_classifier(
    '/content/swahbert',
    label='SwahBERT', num_epochs=5, lr=2e-5
)


In [ ]:
# Fine-tune AfriBERTa-Large
result_afriberta = train_classifier(
    'castorini/afriberta_large',
    label='AfriBERTa-Large', num_epochs=5, lr=2e-5
)


In [ ]:
# Results summary and visualization
all_results = [result_distilbert, result_swahbert, result_afriberta]

results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('history', 'model_path', 'model_name')}
    for r in all_results
])[['tag', 'val_acc', 'val_f1', 'test_acc', 'test_f1', 'test_mcc']]

print('\n' + '='*60)
print('TASK 2 — FINAL RESULTS')
print('='*60)
print(results_df.to_string(index=False))

results_df.to_csv('/content/task2_results.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(results_df['tag'], results_df['test_acc'], color='steelblue')
axes[0].set_title('Test Accuracy by Model', fontweight='bold')
axes[0].set_ylabel('Accuracy'); axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(results_df['tag'], results_df['test_f1'], color='coral')
axes[1].set_title('Test Macro-F1 by Model', fontweight='bold')
axes[1].set_ylabel('Macro-F1'); axes[1].tick_params(axis='x', rotation=20)
axes[2].bar(results_df['tag'], results_df['test_mcc'], color='mediumseagreen')
axes[2].set_title('Test MCC by Model', fontweight='bold')
axes[2].set_ylabel('MCC'); axes[2].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('/content/task2_results.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Experiment Tracking — Logged at the End

The next cell logs all metrics, models, and artifacts to:
- **Weights & Biases**
- **MLflow**


In [ ]:
# Log everything to W&B and MLflow
from getpass import getpass

CONFIG = {
    'task': 'Task 2',
    'optimizer': 'AdamW', 'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
    'grad_clip': 1.0, 'num_classes': num_classes,
    'train_size': len(x_train), 'val_size': len(x_val), 'test_size': len(x_test)
}

os.makedirs('/content/mlruns', exist_ok=True)
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri('file:///content/mlruns')

wandb_key = getpass('Paste W&B API key (wandb.ai/authorize): ')
wandb.login(key=wandb_key)

# ── 1. W&B ────────────────────────────────────────────────────
wandb.init(project='nalapro-swahili', name='task2-bert-finetuning',
           config=CONFIG, reinit=True)
for res in all_results:
    h = res['history']
    for i, ep in enumerate(h['epoch']):
        wandb.log({
            f"{res['tag']}/train_loss": h['train_loss'][i],
            f"{res['tag']}/val_acc":    h['val_acc'][i],
            f"{res['tag']}/val_f1":     h['val_f1'][i],
            'epoch': ep
        })
    wandb.log({
        f"{res['tag']}/test_acc":     res['test_acc'],
        f"{res['tag']}/test_f1":      res['test_f1'],
        f"{res['tag']}/test_mcc":      res['test_mcc'],
        f"{res['tag']}/best_val_acc": res['val_acc'],
        f"{res['tag']}/best_val_f1":  res['val_f1']
    })
    art = wandb.Artifact(f"model-{res['tag']}", type='model')
    art.add_file(res['model_path']); wandb.log_artifact(art)

wandb.log({'task2_summary': wandb.Table(dataframe=results_df)})
wandb.log({'task2_chart':   wandb.Image('/content/task2_results.png')})
wandb_url = wandb.run.url
wandb.finish()

# ── 3. MLflow ────────────────────────────────────────────────
mlflow.set_experiment('nalapro-task2-bert')
for res in all_results:
    with mlflow.start_run(run_name=f"task2-{res['tag']}"):
        mlflow.log_params({**CONFIG, 'model_name': res['model_name'],
                           'epochs': res['epochs'], 'lr': res['lr']})
        h = res['history']
        for i, ep in enumerate(h['epoch']):
            mlflow.log_metric('train_loss', h['train_loss'][i], step=ep)
            mlflow.log_metric('val_acc',    h['val_acc'][i],    step=ep)
            mlflow.log_metric('val_f1',     h['val_f1'][i],     step=ep)
        mlflow.log_metric('test_acc',     res['test_acc'])
        mlflow.log_metric('test_f1',      res['test_f1'])
        mlflow.log_metric('test_mcc',     res['test_mcc'])
        mlflow.log_metric('best_val_acc', res['val_acc'])
        mlflow.log_metric('best_val_f1',  res['val_f1'])
        mlflow.log_artifact(res['model_path'])

with mlflow.start_run(run_name='task2-summary'):
    mlflow.log_artifact('/content/task2_results.csv')
    mlflow.log_artifact('/content/task2_results.png')

print('\n' + '='*60)
print('✓ ALL TRACKING COMPLETE')
print('='*60)
print(f'W&B:    {wandb_url}')
print('Aim:    Run  ! up --repo /content/. ')
print('MLflow: Run  !mlflow ui --backend-store-uri /content/mlruns')
